# CV03 — Few-shot segmentation bằng ảnh hỗ trợ

Support có mask, query chưa có mask lúc suy luận. Học theo episode; lớp novel không trùng lớp train. Prototype là masked average feature, còn cross-attention giữ nhiều support tokens.

Năm cell code, chạy lần lượt. Fixture không phải benchmark; xem REFERENCE.md để chọn dữ liệu thật và paper gốc.


## Cell 1 — Môi trường và seed

Dùng CPU để kiểm tra trước. Tìm repo root từ thư mục notebook, đặt seed và import PyTorch. Không tải pretrained weights.


In [ ]:
from pathlib import Path
import sys
import torch
from torch import nn
import torch.nn.functional as F

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'vision/curriculum').is_dir())
sys.path.insert(0, str(ROOT))
torch.manual_seed(42)
torch.set_num_threads(2)
device = torch.device('cpu')  # Change to cuda after the CPU example works.
print('Device:', device)


## Cell 2 — Tạo batch và model

Support có mask, query chưa có mask lúc suy luận. Học theo episode; lớp novel không trùng lớp train. Prototype là masked average feature, còn cross-attention giữ nhiều support tokens. Dữ liệu trong cell là fixture tổng hợp để bạn đọc được shape; thay bằng Dataset thật trước khi so metric.


In [ ]:
from vision.curriculum.cv03_visual_context import VisualContext, loss_fn
support = torch.rand(2, 1, 3, 32, 32, device=device)
support_mask = torch.zeros(2, 1, 1, 32, 32, device=device)
support_mask[..., 8:24, 8:24] = 1
query = support[:, 0].clone()
target = support_mask[:, 0].clone()
model = VisualContext(mode='prototype').to(device)
batch = (support, support_mask, query, target)


## Cell 3 — Forward và baseline

Đọc shape/output trước khi train. Các số in ra chỉ minh họa phép tính và luồng dự đoán.


In [ ]:
model.eval()
with torch.no_grad():
    logits = model(support, support_mask, query)
print('Query mask logits:', logits.shape)
print('Foreground probability mean:', logits.sigmoid().mean().item())


## Cell 4 — Một bước cập nhật

Tính đúng loss của nhiệm vụ, backward và cập nhật tham số. Một bước không phải quy trình huấn luyện hoàn chỉnh. TENT tự thực hiện update bên trong adapt.


In [ ]:
model.train()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=3e-4)
optimizer.zero_grad(set_to_none=True)
loss = loss_fn(model, *batch)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
print('One-step training loss:', loss.detach().item())


## Cell 5 — Chẩn đoán và hướng nâng cấp

Prototype FG/BG → query-to-support attention. Chấm mIoU theo protocol của bộ dữ liệu; IoU fixture chỉ minh họa công thức.


In [ ]:
model.eval()
with torch.no_grad():
    prediction = model(support, support_mask, query).sigmoid() > 0.5
    truth = target.bool()
    intersection = (prediction & truth).sum()
    union = (prediction | truth).sum()
print('Fixture binary IoU:', (intersection / union.clamp_min(1)).item())
print('Upgrade: VisualContext(mode="cross"); train and evaluate novel-class episodes.')


## Thử nghiệm tiếp theo

Prototype FG/BG → query-to-support attention. Chấm mIoU theo protocol của bộ dữ liệu; IoU fixture chỉ minh họa công thức.

Giữ nguyên split và ngân sách huấn luyện. Đọc [module](../cv03_visual_context.py) và [tài liệu gốc](../REFERENCE.md).
